In [ ]:
import os
import sys
import warnings
import pandas as pd

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.ensemble import RandomForestClassifier

import torch
from sentence_transformers import SentenceTransformer

from scipy.stats import shapiro
from scipy.stats import levene
from scipy.stats import f_oneway

In [ ]:
# Load a pre-trained SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Move the model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
if torch.cuda.is_available():
    print('GPU : ', torch.cuda.get_device_name(0))

# Élimination des avertissements
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore"

In [ ]:
# Stopwords & punctuation
import string
from nltk.corpus import stopwords
punct = string.punctuation
stopw = stopwords.words('english') + [x for x in punct]
stopw += [x.translate
    (str.maketrans('', '', punct)) for x in stopwords.words('english')]

stopw +=  ["'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']

# Tokenizer
from nltk.tokenize import word_tokenize

def tokenize_remove_stopwords(text):
 for token in word_tokenize(text):
    if token in stopw: continue
    yield (token)

**Lecture des données**

In [ ]:
# Lecture du jeu de données et séparation de celles-ci en ensembles d'entraînement et de test
train = pd.read_excel('../data/training_datasets/train_dataset_40pc.xlsx')
test = pd.read_csv('../data/test_dataset_10.csv')

**Entraînement des modèles**

In [ ]:
X_train, y_train = train.text_post.astype('str'), train.category
X_test, y_test = test.text_post.astype('str'), test.category

In [ ]:
vectorizers = {
    # TF-IDF 
    'TfidfVectorizer' : TfidfVectorizer(            
        stop_words=stopw,
        tokenizer=word_tokenize,
        min_df=2,
        token_pattern=None
    ),

    # Word2Vec 

    # Doc2Vec

    # Sentence-BERT
    'SentenceTransformer (all-MiniLM-L6-v2)': FunctionTransformer(
        lambda x: model.encode(
            x.squeeze().astype(str).values,
            batch_size=64,
            convert_to_numpy=True,
            show_progress_bar=True,
            device=device)
    )
}

tf_idf_param_grid = [
    {
        "vectorizer__max_features": [500, 1000, 2500, 5000, 10000, 15000],
        "classify" : [
            LogisticRegression(), 
            LinearSVC(dual="auto"),
            MultinomialNB(),
            ]
    }
]

w2v_d2v_param_grid = [
        {
        "classify" : [
            RandomForestClassifier(), 
            LinearSVC(dual="auto"),
            GaussianNB(),
            ]
    }
]

sbert_param_grid = [
        {
        "classify" : [
            LogisticRegression(), 
            LinearSVC(dual="auto"),
            GaussianNB(),
            ]
    }
]

param_grid = {
    'TfidfVectorizer' : tf_idf_param_grid,
    'SentenceTransformer (all-MiniLM-L6-v2)' : sbert_param_grid
}

In [189]:
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=0)

results = []

for vectorizer_name, vectorizer in vectorizers.items():
    specific_param_grid = param_grid.get(vectorizer_name, {})

    pipeline = Pipeline([
        ("vectorizer", vectorizer),
        ("classify", "passthrough")
    ])

    grid_search = GridSearchCV(
        pipeline, 
        param_grid=specific_param_grid, 
        verbose=5, 
        cv=cv,
        refit='f1_macro', 
        scoring=['accuracy','f1_macro']
    )

    print(f'Running GridSearchCV for {vectorizer_name}...')
    grid_search.fit(X_train, y_train)

    # Stocker les résultats
    results_dic = grid_search.cv_results_
    results_dic['Vectorizer'] = vectorizer_name
    results.append(results_dic)

Running GridSearchCV for TfidfVectorizer...
Fitting 30 folds for each of 18 candidates, totalling 540 fits
[CV 1/30] END classify=LogisticRegression(), vectorizer__max_features=500; accuracy: (test=0.780) f1_macro: (test=0.738) total time=   0.6s
[CV 2/30] END classify=LogisticRegression(), vectorizer__max_features=500; accuracy: (test=0.780) f1_macro: (test=0.743) total time=   0.4s
[CV 3/30] END classify=LogisticRegression(), vectorizer__max_features=500; accuracy: (test=0.730) f1_macro: (test=0.653) total time=   0.4s
[CV 4/30] END classify=LogisticRegression(), vectorizer__max_features=500; accuracy: (test=0.820) f1_macro: (test=0.777) total time=   0.5s
[CV 5/30] END classify=LogisticRegression(), vectorizer__max_features=500; accuracy: (test=0.750) f1_macro: (test=0.693) total time=   0.5s
[CV 6/30] END classify=LogisticRegression(), vectorizer__max_features=500; accuracy: (test=0.750) f1_macro: (test=0.705) total time=   0.5s
[CV 7/30] END classify=LogisticRegression(), vectoriz

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

[CV 1/30] END classify=LogisticRegression(); accuracy: (test=0.780) f1_macro: (test=0.747) total time=  18.8s


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

...

[CV 29/30] END classify=GaussianNB(); accuracy: (test=0.840) f1_macro: (test=0.824) total time=  22.2s


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

[CV 30/30] END classify=GaussianNB(); accuracy: (test=0.780) f1_macro: (test=0.755) total time=  24.2s


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [195]:
results_df = pd.concat([pd.DataFrame(results[0]), pd.DataFrame(results[1])])
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classify,param_vectorizer__max_features,params,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,...,split24_test_f1_macro,split25_test_f1_macro,split26_test_f1_macro,split27_test_f1_macro,split28_test_f1_macro,split29_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro,Vectorizer
0,0.754934,0.559408,0.078402,0.047307,LogisticRegression(),500.0,"{'classify': LogisticRegression(), 'vectorizer...",0.78,0.78,0.73,...,0.715100,0.684313,0.705154,0.652115,0.706015,0.705154,0.708280,0.041501,16,TfidfVectorizer
1,0.507763,0.064611,0.055327,0.009840,LogisticRegression(),1000.0,"{'classify': LogisticRegression(), 'vectorizer...",0.78,0.78,0.75,...,0.742871,0.677579,0.710346,0.733519,0.764336,0.681566,0.715356,0.038051,15,TfidfVectorizer
2,0.495741,0.021409,0.054087,0.007962,LogisticRegression(),2500.0,"{'classify': LogisticRegression(), 'vectorizer...",0.77,0.79,0.76,...,0.714286,0.686520,0.705154,0.728742,0.760684,0.690476,0.718021,0.038659,13,TfidfVectorizer
3,0.577951,0.203556,0.057898,0.015767,LogisticRegression(),5000.0,"{'classify': LogisticRegression(), 'vectorizer...",0.78,0.78,0.76,...,0.714286,0.670218,0.681566,0.738095,0.760684,0.675442,0.718111,0.042167,10,TfidfVectorizer
4,0.600321,0.158758,0.064434,0.036450,LogisticRegression(),10000.0,"{'classify': LogisticRegression(), 'vectorizer...",0.78,0.78,0.76,...,0.714286,0.670218,0.681566,0.738095,0.760684,0.675442,0.718111,0.042167,10,TfidfVectorizer
5,0.493710,0.021504,0.055921,0.010300,LogisticRegression(),15000.0,"{'classify': LogisticRegression(), 'vectorizer...",0.78,0.78,0.76,...,0.714286,0.670218,0.681566,0.738095,0.760684,0.675442,0.718111,0.042167,10,TfidfVectorizer
6,0.503996,0.045269,0.056122,0.009949,LinearSVC(),500.0,"{'classify': LinearSVC(), 'vectorizer__max_fea...",0.77,0.68,0.73,...,0.689955,0.683401,0.665775,0.604167,0.648352,0.696181,0.687567,0.045831,18,TfidfVectorizer
7,0.535246,0.059095,0.058404,0.012506,LinearSVC(),1000.0,"{'classify': LinearSVC(), 'vectorizer__max_fea...",0.75,0.73,0.74,...,0.726776,0.652778,0.661535,0.642707,0.717882,0.736264,0.715823,0.048944,14,TfidfVectorizer
8,0.544385,0.078349,0.062785,0.018830,LinearSVC(),2500.0,"{'classify': LinearSVC(), 'vectorizer__max_fea...",0.74,0.74,0.76,...,0.776000,0.733333,0.687197,0.739583,0.708769,0.717882,0.738448,0.037436,8,TfidfVectorizer
9,0.526075,0.110106,0.057309,0.012246,LinearSVC(),5000.0,"{'classify': LinearSVC(), 'vectorizer__max_fea...",0.74,0.77,0.77,...,0.776000,0.739583,0.687197,0.751915,0.727044,0.741892,0.740288,0.038565,5,TfidfVectorizer


In [198]:
results_df = results_df.sort_values(by=["rank_test_f1_macro"])
results_df[['Vectorizer', 'param_classify', 'param_vectorizer__max_features', 'mean_test_f1_macro', 'std_test_f1_macro', 'rank_test_f1_macro']]

,Vectorizer,param_classify,param_vectorizer__max_features,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro
0,SentenceTransformer (all-MiniLM-L6-v2),LogisticRegression(),NaN,0.816303,0.041909,1
17,TfidfVectorizer,MultinomialNB(),15000.0,0.758443,0.039299,1
16,TfidfVectorizer,MultinomialNB(),10000.0,0.758443,0.039299,1
15,TfidfVectorizer,MultinomialNB(),5000.0,0.758443,0.039299,1
2,SentenceTransformer (all-MiniLM-L6-v2),GaussianNB(),NaN,0.807451,0.037885,2
1,SentenceTransformer (all-MiniLM-L6-v2),LinearSVC(),NaN,0.801113,0.046115,3
14,TfidfVectorizer,MultinomialNB(),2500.0,0.752043,0.038500,4
11,TfidfVectorizer,LinearSVC(),15000.0,0.740288,0.038565,5
9,TfidfVectorizer,LinearSVC(),5000.0,0.740288,0.038565,5
10,TfidfVectorizer,LinearSVC(),10000.0,0.740288,0.038565,5


**1. Modèles TF-IDF**

Nous allons analyser l'effet de trois variables sur la performance des modèles de classification :  
1. Le nombre de traits discriminants 
2. Le type de classifieur utilisé 
3. Le ratio de données incels sur la performance des modèles

Nous retiendrons ensuite les trois meilleurs modèles pour la suite des analyses.

In [201]:
modeles_tf_idf = results_df[results_df['Vectorizer'] == 'TfidfVectorizer']
modeles_tf_idf.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classify,param_vectorizer__max_features,params,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,...,split24_test_f1_macro,split25_test_f1_macro,split26_test_f1_macro,split27_test_f1_macro,split28_test_f1_macro,split29_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro,Vectorizer
17,0.488499,0.042777,0.055677,0.010154,MultinomialNB(),15000.0,"{'classify': MultinomialNB(), 'vectorizer__max...",0.75,0.79,0.76,...,0.78022,0.688057,0.747243,0.777184,0.829060,0.760684,0.758443,0.039299,1,TfidfVectorizer
16,0.476385,0.023541,0.054900,0.009860,MultinomialNB(),10000.0,"{'classify': MultinomialNB(), 'vectorizer__max...",0.75,0.79,0.76,...,0.78022,0.688057,0.747243,0.777184,0.829060,0.760684,0.758443,0.039299,1,TfidfVectorizer
15,0.484949,0.033240,0.055511,0.013917,MultinomialNB(),5000.0,"{'classify': MultinomialNB(), 'vectorizer__max...",0.75,0.79,0.76,...,0.78022,0.688057,0.747243,0.777184,0.829060,0.760684,0.758443,0.039299,1,TfidfVectorizer
14,0.487855,0.048168,0.053217,0.008449,MultinomialNB(),2500.0,"{'classify': MultinomialNB(), 'vectorizer__max...",0.74,0.80,0.79,...,0.78022,0.701294,0.724265,0.745547,0.819086,0.727044,0.752043,0.038500,4,TfidfVectorizer
11,0.507346,0.055459,0.055337,0.011071,LinearSVC(),15000.0,"{'classify': LinearSVC(), 'vectorizer__max_fea...",0.74,0.77,0.77,...,0.77600,0.739583,0.687197,0.751915,0.727044,0.741892,0.740288,0.038565,5,TfidfVectorizer


In [205]:
# Colonnes des scores individuels F1_macro
score_columns = [col for col in modeles_tf_idf.columns if "split" in col and "test_f1_macro" in col]

# Transformation des données en format long
long_format = pd.melt(
    modeles_tf_idf,
    id_vars=["param_classify", "param_vectorizer__max_features"],
    value_vars=score_columns,
    var_name="Split",
    value_name="test_f1_macro"
)

# Création d'une colonne unique pour chaque configuration (modèle + features)
long_format["Model_Features"] = (
    long_format["param_classify"].astype(str) 
    + "_" 
    + long_format["param_vectorizer__max_features"].astype(str)
)

# Aperçu des données transformées
long_format.head()

,param_classify,param_vectorizer__max_features,Split,test_f1_macro,Model_Features
0,MultinomialNB(),15000.0,split0_test_f1_macro,0.699483,MultinomialNB()_15000.0
1,MultinomialNB(),10000.0,split0_test_f1_macro,0.699483,MultinomialNB()_10000.0
2,MultinomialNB(),5000.0,split0_test_f1_macro,0.699483,MultinomialNB()_5000.0
3,MultinomialNB(),2500.0,split0_test_f1_macro,0.684313,MultinomialNB()_2500.0
4,LinearSVC(),15000.0,split0_test_f1_macro,0.701287,LinearSVC()_15000.0


ANOVA - Vérification des conditions

1. Indépendance

Cette condition est remplie par la validation croisée où chaque pli est indépendant des autres

*2. Distribution normale des données*

In [ ]:
liste_donnees = [] 
for model in long_format['Model_Features'].unique():
    # Exemple de données pour un groupe
    donnees = long_format[long_format['Model_Features'] == model]['test_f1_macro'].tolist()

    stat, p = shapiro(donnees)

    if p < 0.05:
        print(model, "Les données ne suivent pas une distribution normale.")
        print(f"Statistique de Shapiro-Wilk : {stat}, p-value : {p}")

    liste_donnees.append(donnees)

*3. Homogénéité des variances entre les groupes*

In [209]:
stat, p = levene(
    *liste_donnees
)
print(f"Statistique de Levene : {stat}, p-value : {p}")

if p > 0.05:
    print("Les variances sont homogènes.")
else:
    print("Les variances ne sont pas homogènes.")


Statistique de Levene : 0.2902948513934167, p-value : 0.9978208036812601
Les variances sont homogènes.


**ANOVA**   
Comme les conditions sont respectées, nous allons utiliser un test ANOVA pour comparer les moyennes de performances entre nos modèles.

*Anova à deux facteurs*: (modifier pour ajouter un troisième facteur plus tard)
1. Classifieur utilisé (Multinomial Naive Bayes, Logistic Regression, Support Vector Machine)
2. Nombre de features (5000, 10 000, 15 000)

In [210]:
long_format

,param_classify,param_vectorizer__max_features,Split,test_f1_macro,Model_Features
0,MultinomialNB(),15000.0,split0_test_f1_macro,0.699483,MultinomialNB()_15000.0
1,MultinomialNB(),10000.0,split0_test_f1_macro,0.699483,MultinomialNB()_10000.0
2,MultinomialNB(),5000.0,split0_test_f1_macro,0.699483,MultinomialNB()_5000.0
3,MultinomialNB(),2500.0,split0_test_f1_macro,0.684313,MultinomialNB()_2500.0
4,LinearSVC(),15000.0,split0_test_f1_macro,0.701287,LinearSVC()_15000.0
...,...,...,...,...,...
535,LinearSVC(),1000.0,split29_test_f1_macro,0.736264,LinearSVC()_1000.0
536,LogisticRegression(),1000.0,split29_test_f1_macro,0.681566,LogisticRegression()_1000.0
537,LogisticRegression(),500.0,split29_test_f1_macro,0.705154,LogisticRegression()_500.0
538,MultinomialNB(),500.0,split29_test_f1_macro,0.710346,MultinomialNB()_500.0


In [250]:
modeles_tf_idf[modeles_tf_idf['param_vectorizer__max_features'] == 15000]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classify,param_vectorizer__max_features,params,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,...,split24_test_f1_macro,split25_test_f1_macro,split26_test_f1_macro,split27_test_f1_macro,split28_test_f1_macro,split29_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro,Vectorizer
17,0.488499,0.042777,0.055677,0.010154,MultinomialNB(),15000.0,"{'classify': MultinomialNB(), 'vectorizer__max...",0.75,0.79,0.76,...,0.780220,0.688057,0.747243,0.777184,0.829060,0.760684,0.758443,0.039299,1,TfidfVectorizer
11,0.507346,0.055459,0.055337,0.011071,LinearSVC(),15000.0,"{'classify': LinearSVC(), 'vectorizer__max_fea...",0.74,0.77,0.77,...,0.776000,0.739583,0.687197,0.751915,0.727044,0.741892,0.740288,0.038565,5,TfidfVectorizer
5,0.493710,0.021504,0.055921,0.010300,LogisticRegression(),15000.0,"{'classify': LogisticRegression(), 'vectorizer...",0.78,0.78,0.76,...,0.714286,0.670218,0.681566,0.738095,0.760684,0.675442,0.718111,0.042167,10,TfidfVectorizer


In [224]:
# Importing libraries 
import statsmodels.api as sm 
from statsmodels.formula.api import ols 
  
# Performing two-way ANOVA 
model = ols( 
    'test_f1_macro ~ C(param_classify) + C(param_vectorizer__max_features) +C(param_classify):C(param_vectorizer__max_features)', 
    data=long_format).fit() 


anova_results = sm.stats.anova_lm(model, typ=2) 

# Format the p-values with several decimals
anova_results['reject H0'] = anova_results['PR(>F)'].apply(lambda x: True if x < 0.05 else False)

anova_results

,sum_sq,df,F,PR(>F),reject H0
C(param_classify),0.053432,2.0,15.450091,3.028336e-07,True
C(param_vectorizer__max_features),0.146054,5.0,16.892964,1.718903e-15,True
C(param_classify):C(param_vectorizer__max_features),0.048763,10.0,2.820042,2.035184e-03,True
Residual,0.902626,522.0,NaN,NaN,False


**Test de Tukey (HSD)**

Le résultat au test ANOVA indique que les deux paramètres (classifieur et nombre de traits discriminants) ont un effet significatif (p < 0.05) sur la performance des modèles (f1_macro).  
Nous allons donc maintenant analyser les modèles en paires (comparaisons 1 à 1) pour explorer plus en détail leurs différences. 

In [225]:
from scipy.stats import tukey_hsd
import re

# Exécuter le test de Tukey
tukey_result = tukey_hsd(*liste_donnees)

# Renommer les indices pour les modèles
model_names = {
    i : long_format['Model_Features'].unique()[i] for i in range(len(long_format['Model_Features'].unique()))
}

In [246]:
str_results = '\n'.join(str(tukey_result).split('\n')[1:])
str_results = re.sub(r'\s-\s', '-', str_results)
str_results = str_results.replace('Lower CI', 'Lower_CI').replace('Upper CI', 'Upper_CI').replace('(', '').replace(')', '')

with open('../results/tukey_results_tf-idf_models.txt', 'w') as f:
    f.write(str_results)

df = pd.read_csv('../results/tukey_results_tf-idf_models.txt', delim_whitespace=True, header=0)
df[['Model 1', 'Model 2']] = df['Comparison'].str.split('-', n=1, expand=True)

df['Model 1'] = df['Model 1'].astype(int).map(model_names)
df['Model 2'] = df['Model 2'].astype(int).map(model_names)

df['Reject H0'] = df['p-value'].apply(lambda x: True if x < 0.05 else False)

df = df[['Model 1', 'Model 2', 'Statistic', 'p-value', 'Lower_CI', 'Upper_CI', 'Reject H0']]
df.to_csv('../results/tukey_results_tf-idf_models.csv', index=False)

In [247]:
df = df[df['Model 1'].str.contains('MultinomialNB()')]
df = df[df['Model 2'].str.contains('MultinomialNB()')]

df['n_features_model1'] = df['Model 1'].apply(lambda x: int(float(x.split('_')[1])))
df['n_features_model2'] = df['Model 2'].apply(lambda x: int(float(x.split('_')[1])))

df = df[['n_features_model1', 'n_features_model2', 'p-value', 'Reject H0']]
df.sort_values(by=['n_features_model1', 'n_features_model2'])

,n_features_model1,n_features_model2,p-value,Reject H0
280,500,1000,0.030,True
275,500,2500,0.000,True
274,500,5000,0.000,True
273,500,10000,0.000,True
272,500,15000,0.000,True
151,1000,500,0.030,True
139,1000,2500,0.659,False
138,1000,5000,0.240,False
137,1000,10000,0.240,False
136,1000,15000,0.240,False


La comparaison en paire permet d'observer qu'au-delà le 1000 traits discriminants, les différences de performances entre les modèles ne sont plus statistiquement significatives.   
Nous conserverons donc un maximum de 1000 traits. 

___